## Setup Connection

In [105]:
import requests
import time
import dotenv
import os
import base64
import webbrowser
from IPython.display import display, HTML

In [106]:
# Startup

dotenv.load_dotenv("./../../secrets.env")

APP_KEY = os.getenv("SAXO_TESTING_APP_KEY")
REDIRECT_URL = os.getenv("REDIRECTION_URI")
APP_SECRET = os.getenv("SAXO_TESTING_APP_SECRET1")
TOKEN_URL = "https://sim.logonvalidation.net/token"
API_BASE_URL = "https://gateway.saxobank.com/sim/openapi"
AUTH_BASE_URL = "https://sim.logonvalidation.net/authorize"

setup_url = f"https://sim.logonvalidation.net/authorize?response_type=code&client_id={APP_KEY}&redirect_uri=http://localhost:2000&state=12345"

if (APP_KEY is None or REDIRECT_URL is None or APP_KEY is None):
    raise ValueError("Not all secrets have been found")

In [107]:
setup_url

'https://sim.logonvalidation.net/authorize?response_type=code&client_id=5a0d8e4d26e94bdf813938dc21978d27&redirect_uri=http://localhost:2000&state=12345'

In [108]:
p = requests.Request('GET', setup_url).prepare()
print(f"Please log in here:\n{p.url}\n")
webbrowser.open(p.url)

Please log in here:
https://sim.logonvalidation.net/authorize?response_type=code&client_id=5a0d8e4d26e94bdf813938dc21978d27&redirect_uri=http://localhost:2000&state=12345



True

In [109]:
redirected_url = input("Paste the full URL you were redirected to: ")
no_beginning = redirected_url[redirected_url.index("code=")+5:]
auth_code = no_beginning[:no_beginning.index("&")]

In [110]:
auth_str = f"{APP_KEY}:{APP_SECRET}"
base64_auth = base64.b64encode(auth_str.encode()).decode()

headers = {
    "Authorization": f"Basic {base64_auth}",
    "Content-Type": "application/x-www-form-urlencoded"
}

payload = {
    "grant_type": "authorization_code",
    "code": auth_code,
    "redirect_uri": REDIRECT_URL
}

In [111]:
print("Exchanging code for tokens...")
response = requests.post(TOKEN_URL, headers=headers, data=payload)

Exchanging code for tokens...


In [112]:
token_data = response.json()

In [113]:
url = f"{API_BASE_URL}/port/v1/users/me"
access_token = token_data.get("access_token")
headers = {
    "Authorization": f"Bearer {access_token}"
}

response = requests.get(url, headers=headers)

In [114]:
response.status_code

200

In [115]:
response.json()

{'Active': True,
 'ClientKey': '4|10kdB0NSWjosZ5I2-I2Q==',
 'Culture': 'en-GB',
 'Language': 'en',
 'LastLoginStatus': 'Successful',
 'LastLoginTime': '2026-05-03T15:46:52.120000Z',
 'LegalAssetTypes': ['FxSpot',
  'FxForwards',
  'ContractFutures',
  'Stock',
  'StockOption',
  'Bond',
  'FuturesOption',
  'StockIndexOption',
  'Cash',
  'CfdOnStock',
  'CfdOnIndex',
  'StockIndex',
  'CfdOnEtf',
  'CfdOnEtc',
  'CfdOnEtn',
  'CfdOnFund',
  'CfdOnRights',
  'CfdOnCompanyWarrant',
  'Etf',
  'Etc',
  'Etn',
  'Fund',
  'FxSwap',
  'Rights',
  'IpoOnStock',
  'CompanyWarrant'],
 'MarketDataViaOpenApiTermsAccepted': True,
 'Name': 'Robert Fischer',
 'TimeZoneId': 26,
 'UserId': '22169168',
 'UserKey': '4|10kdB0NSWjosZ5I2-I2Q=='}

In [116]:
def renew_token_data(token_data):


    auth_str = f"{APP_KEY}:{APP_SECRET}"

    converted_auth_str = base64.b64encode(auth_str.strip().encode("utf-8")).decode("utf-8")

    headers = {
        "Authorization": f"Basic {converted_auth_str}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    payload = {
        "grant_type": "refresh_token",
        "refresh_token": token_data.get("refresh_token"),
        "redirect_uri": REDIRECT_URL
    }

    response = requests.post(TOKEN_URL, headers=headers, data=payload)

    token_data = response.json()

    return token_data

In [117]:
def make_api_request_selector(request_obj, token_data):
    if(request_obj.get("Type") == "GET"):
        access_token = token_data.get('access_token')
        return requests.get(
            request_obj.get("URL"), 
            headers={
                "Authorization": f"Bearer {access_token}"
            }
    )
    elif(request_obj.get("Type") == "POST"):
        return requests.post(
            request_obj.get("URL"), 
            headers={
                "Authorization": f"Bearer {token_data.get('access_token')}",
                "Content-Type": "application/json"
            }
        )

    

In [118]:
def make_api_request(request_obj, token_data):

    response = make_api_request_selector(request_obj=request_obj, token_data=token_data)

    if(response.ok):
        return response
    else:
        print("Trying to get new Token")
        token_data = renew_token_data(token_data=token_data)
        response = make_api_request_selector(request_obj=request_obj, token_data=token_data)
        return response
        

In [119]:
response.status_code

200

## Get info about the user

In [120]:
request_obj = {
    "Type": "GET",
    "URL": f"{API_BASE_URL}/port/v1/balances/me"
}

response = make_api_request(request_obj=request_obj, token_data=token_data)
response.json()

{'CalculationReliability': 'Ok',
 'CashBalance': 1000000.0,
 'CashBlocked': 0.0,
 'CashBlockedFromWithdrawal': 0.0,
 'ChangesScheduled': False,
 'ClosedPositionsCount': 0,
 'CollateralAvailable': 1000000.0,
 'CollateralCreditValue': {'Line': 1000000.0, 'UtilizationPct': 0.0},
 'CorporateActionUnrealizedAmounts': 0.0,
 'CostToClosePositions': 0.0,
 'Currency': 'EUR',
 'CurrencyDecimals': 2,
 'FinancingAccruals': 0.0,
 'InitialMargin': {'CollateralAvailable': 1000000.0,
  'CollateralCreditValue': {'Line': 1000000.0, 'UtilizationPct': 0.0},
  'MarginAvailable': 1000000.0,
  'MarginCollateralNotAvailable': 0.0,
  'MarginCollateralNotAvailableIncludingCostToClosePositions': 0.0,
  'MarginUsedByCurrentPositions': 0.0,
  'MarginUtilizationPct': 0.0,
  'NetEquityForMargin': 1000000.0,
  'OtherCollateralDeduction': 0.0},
 'IsPortfolioMarginModelSimple': True,
 'MarginAndCollateralUtilizationPct': 0.0,
 'MarginAvailableForTrading': 1000000.0,
 'MarginCollateralNotAvailable': 0.0,
 'MarginCollate

## Get list of stocks
First we need to get the account key

In [140]:
request_obj = {
    "Type": "GET",
    "URL": f"{API_BASE_URL}/port/v1/accounts/me"
}

response = make_api_request(request_obj=request_obj, token_data=token_data)
AccountKey = response.json().get("Data")[0].get("AccountKey")

In [141]:
request_obj = {
    "Type": "GET",
    "URL": f"{API_BASE_URL}/ref/v1/instruments?AssetTypes=Stock&$top=1000&$skip=0&AccountKey={AccountKey}&IncludeNonTradable=False"
}

response = make_api_request(request_obj=request_obj, token_data=token_data)


In [146]:
response.json()["Data"][0]

{'AssetType': 'Stock',
 'CurrencyCode': 'DKK',
 'Description': 'Novo Nordisk B A/S',
 'ExchangeId': 'CSE',
 'GroupId': 4552,
 'Identifier': 15629,
 'IssuerCountry': 'DK',
 'PrimaryListing': 15629,
 'SummaryType': 'Instrument',
 'Symbol': 'NOVOb:xcse',
 'TradableAs': ['Stock']}